# patches

> Cut full-resolution image/mask pairs into overlapping training patches. Generic over the number of source channels and the number of mask classes.

The disk layout is modelled on the source layout: when there is a single source channel, image and mask patches share the same base name (`{id}.png` in `images/` and `masks/` respectively). For multi-channel sources each channel is saved as `{id}_ch{i}.png`. The model input image is assembled later (see `channels`), so the same patches feed binary or multi-class tasks.

In [ ]:
#| default_exp patches

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import cv2
import numpy as np

from segmentation_tools_pt.masks import binarize, fit_rotated_rects_multiclass

## Patch configuration

One dataclass holds every knob for patch extraction. Overlap (`stride < patch_size`) ensures no target sits permanently on a hard patch boundary.

In [ ]:
#| export
@dataclass
class PatchConfig:
    "Knobs controlling how full images are cut into patches."
    patch_size: int = 400          # side length of a saved patch (px)
    stride: int = 200              # step between patch centres (< patch_size -> overlap)
    min_fg_pixels: int = 500       # patches below this foreground count are 'empty'
    val_fraction: float = 0.2      # fraction of SOURCE images held out for validation
    max_empty_fraction: float = 0.15  # probability of keeping an empty patch
    seed: int = 42                 # base RNG seed

## Grid of patch centres

First job: given an image size, where do the patch centres go? We step by `stride` and always add a final anchor flush with the right/bottom edge so the border is covered.

In [ ]:
# experiment
h, w, patch, stride = 1000, 1200, 400, 200
half = patch // 2
ys = list(range(half, h - half + 1, stride))
if ys[-1] < h - half: ys.append(h - half)
print('row centres:', ys)

In [ ]:
#| export
def grid_centers(
    h: int, w: int,        # image height and width
    patch_size: int,       # patch side length
    stride: int,           # step between centres
) -> List[Tuple[int, int]]:
    """Return `(cy, cx)` patch centres tiling an `h`x`w` image, edges included.

    Example:
        >>> grid_centers(8, 8, 4, 4)
        [(2, 2), (2, 6), (6, 2), (6, 6)]
    """
    half = patch_size // 2
    ys = list(range(half, h - half + 1, stride)) or [half]
    xs = list(range(half, w - half + 1, stride)) or [half]
    if ys[-1] < h - half: ys.append(h - half)
    if xs[-1] < w - half: xs.append(w - half)
    return [(cy, cx) for cy in ys for cx in xs]

In [ ]:
# test
c = grid_centers(8, 8, 4, 4)
assert c == [(2, 2), (2, 6), (6, 2), (6, 6)]
# overlap case still covers the far edge
assert grid_centers(10, 10, 4, 4)[-1] == (8, 8)

## Crop one patch

Second job: extract a square crop centred at `(cy, cx)`. Works for 2-D (mask) and 3-D (multi-channel) arrays.

In [ ]:
#| export
def crop_patch(
    arr: np.ndarray,       # (H, W) or (H, W, C) array
    cy: int, cx: int,      # patch centre
    size: int,             # patch side length
) -> np.ndarray:
    """Crop a `size`x`size` window centred at `(cy, cx)`.

    Example:
        >>> crop_patch(np.arange(64).reshape(8, 8), 2, 2, 4).shape
        (4, 4)
    """
    half = size // 2
    y0, x0 = cy - half, cx - half
    return arr[y0:y0 + size, x0:x0 + size]

In [ ]:
# test
assert crop_patch(np.arange(64).reshape(8, 8), 2, 2, 4).shape == (4, 4)
assert crop_patch(np.zeros((8, 8, 3)), 4, 4, 4).shape == (4, 4, 3)

## Per-class pixel counts

Third job: how much of each class is in a mask patch? Used to flag empty patches and to weight sampling later.

In [ ]:
#| export
def count_class_pixels(
    mask: np.ndarray,   # (H, W) integer-label mask
    num_classes: int,   # number of foreground classes K
) -> Dict[int, int]:
    """Count pixels per foreground label `1..K`.

    Example:
        >>> count_class_pixels(np.array([[0, 1], [2, 2]]), 2)
        {1: 1, 2: 2}
    """
    return {c: int((mask == c).sum()) for c in range(1, num_classes + 1)}

In [ ]:
# test
assert count_class_pixels(np.array([[0, 1], [2, 2]]), 2) == {1: 1, 2: 2}

## Extract all patches from one sample

The composing function. It reads the source channels and mask, optionally snaps mask blobs to rectangles, tiles the grid, keeps every patch with enough foreground (and a sampled fraction of empty ones), writes them out, and returns per-patch metadata. Reproducible per sample via a seed derived from `(seed, sample_id)`.

`mask_mode` mirrors `dataset.py`'s `SegmentationDataset`: `'single'` (default, unchanged -- one integer-label mask, `fit_rects` supported) writes `{pid}.png`. `'stacked'` reads one `mask_channels`-channel PNG (`mask_path`) and writes one K-channel PNG per patch. `'per_class_files'` reads `mask_paths`/`class_names` (K separate single-channel files) and writes `{pid}_{class_name}.png` per patch. Both multilabel modes keep every channel independently binarized, so overlap (void inside chip) survives patching; `fit_rects` is not yet supported for them (Phase 3 scope is patch I/O, not per-channel rectangle-fitting).

In [ ]:
#| export
def extract_patches_from_sample(
    sample_id: str,                  # unique id for the source image
    channel_paths: Sequence[str],    # paths to the source single-channel images
    mask_path: str,                  # path to the integer-label ('single') or stacked K-channel ('stacked') mask; None for 'per_class_files'
    output_dir: Path,                # split dir; gets images/ and masks/ subdirs
    cfg: PatchConfig,                # extraction settings
    num_classes: int = 1,            # number of foreground classes (mask_mode='single' only)
    fit_rects: bool = False,         # snap mask blobs to rotated rectangles first (mask_mode='single' only)
    *,
    mask_mode: str = 'single',                    # 'single' | 'stacked' | 'per_class_files'
    mask_channels: int = None,                    # K, required for 'stacked'
    mask_paths: Sequence[str] = None,              # K per-class mask paths, required for 'per_class_files'
    class_names: Sequence[str] = None,             # K class name suffixes, required for 'per_class_files'
) -> List[Dict]:
    """Tile one source image into patches on disk and return their metadata.

    For a single source channel the image patch is saved as `{pid}.png` so that
    image and mask filenames match. Multi-channel sources still use `{pid}_ch{i}.png`.
    """
    assert mask_mode in ('single', 'stacked', 'per_class_files'), f'unknown mask_mode {mask_mode!r}'
    if mask_mode == 'stacked':
        assert mask_channels is not None, "mask_mode='stacked' requires mask_channels"
    if mask_mode == 'per_class_files':
        assert mask_paths is not None and class_names is not None and len(mask_paths) == len(class_names), \
            "mask_mode='per_class_files' requires mask_paths and class_names of equal length"
    if fit_rects:
        assert mask_mode == 'single', "fit_rects is not yet supported for multilabel mask_mode"

    output_dir = Path(output_dir)
    (output_dir / 'images').mkdir(parents=True, exist_ok=True)
    (output_dir / 'masks').mkdir(parents=True, exist_ok=True)

    channels = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) for p in channel_paths]
    for arr, p in zip(channels, channel_paths):
        assert arr is not None, f'Cannot read channel: {p}'

    multilabel = mask_mode != 'single'
    if mask_mode == 'per_class_files':
        chans_m = []
        for p in mask_paths:
            m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
            assert m is not None, f'Cannot read mask: {p}'
            chans_m.append((m > 127).astype(np.uint8))
        mask = np.stack(chans_m, axis=-1)  # (H, W, K) {0,1}
    elif mask_mode == 'stacked':
        arr = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
        assert arr is not None, f'Cannot read mask: {mask_path}'
        if arr.ndim == 2:
            arr = arr[:, :, None]
        mask = (arr[:, :, :mask_channels] > 127).astype(np.uint8)  # (H, W, K) {0,1}
    else:
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        assert mask is not None, f'Cannot read mask: {mask_path}'
        mask = binarize(mask) if num_classes == 1 else mask.astype(np.int32)
        if fit_rects:
            mask = fit_rotated_rects_multiclass(mask, max(num_classes, 1))

    # Deterministic per-sample RNG: same run -> same patches, different sources differ.
    rng = np.random.default_rng(abs(hash((cfg.seed, sample_id))) % (2 ** 32))
    h, w = channels[0].shape
    single_channel = len(channels) == 1
    meta: List[Dict] = []
    for cy, cx in grid_centers(h, w, cfg.patch_size, cfg.stride):
        mpatch = crop_patch(mask, cy, cx, cfg.patch_size)  # crop_patch is shape-agnostic: (H,W) or (H,W,K)
        fg = int(mpatch.sum()) if multilabel else int((mpatch > 0).sum())
        is_empty = fg < cfg.min_fg_pixels
        if is_empty and rng.random() > cfg.max_empty_fraction:
            continue
        pid = f'{sample_id}_y{cy:05d}_x{cx:05d}'
        if single_channel:
            cv2.imwrite(str(output_dir / 'images' / f'{pid}.png'), crop_patch(channels[0], cy, cx, cfg.patch_size))
        else:
            for i, ch in enumerate(channels):
                cv2.imwrite(str(output_dir / 'images' / f'{pid}_ch{i}.png'), crop_patch(ch, cy, cx, cfg.patch_size))

        if mask_mode == 'per_class_files':
            for k, name in enumerate(class_names):
                cv2.imwrite(str(output_dir / 'masks' / f'{pid}_{name}.png'), (mpatch[:, :, k] * 255).astype(np.uint8))
            class_pixels = {name: int(mpatch[:, :, k].sum()) for k, name in enumerate(class_names)}
        elif mask_mode == 'stacked':
            cv2.imwrite(str(output_dir / 'masks' / f'{pid}.png'), (mpatch * 255).astype(np.uint8))
            class_pixels = {k: int(mpatch[:, :, k].sum()) for k in range(mask_channels)}
        else:
            out_mask = (mpatch * 255).astype(np.uint8) if num_classes == 1 else mpatch.astype(np.uint8)
            cv2.imwrite(str(output_dir / 'masks' / f'{pid}.png'), out_mask)
            class_pixels = count_class_pixels(mpatch, max(num_classes, 1))

        meta.append({
            'sample_id': pid, 'source_id': sample_id,
            'center_y': cy, 'center_x': cx,
            'fg_pixels': fg, 'is_empty': is_empty,
            'class_pixels': class_pixels,
        })
    return meta

In [ ]:
# test: single-channel patches share image/mask names; multi-channel keeps _ch{i} channels
import tempfile
rng = np.random.default_rng(0)
with tempfile.TemporaryDirectory() as d:
    d = Path(d)

    # --- single-channel layout: image and mask filenames must match ---
    (d / 'single' / 'images').mkdir(parents=True)
    (d / 'single' / 'masks').mkdir(parents=True)
    img = rng.integers(0, 256, (40, 40), np.uint8)
    mask = np.zeros((40, 40), np.uint8); mask[10:30, 10:30] = 255
    cv2.imwrite(str(d / 'single' / 'images' / 'a.png'), img)
    cv2.imwrite(str(d / 'single' / 'masks' / 'a.png'), mask)
    cfg = PatchConfig(patch_size=20, stride=20, min_fg_pixels=1, max_empty_fraction=0.0)
    meta = extract_patches_from_sample('a', [d / 'single' / 'images' / 'a.png'], d / 'single' / 'masks' / 'a.png', d / 'single_out', cfg)
    assert len(meta) >= 1
    img_names = sorted(p.name for p in (d / 'single_out' / 'images').glob('*.png'))
    mask_names = sorted(p.name for p in (d / 'single_out' / 'masks').glob('*.png'))
    assert img_names == mask_names, f'mismatch: {img_names} vs {mask_names}'
    assert not any('_ch0' in n or '_mask' in n for n in img_names)
    assert all(m['fg_pixels'] >= 1 for m in meta)

    # --- multi-channel layout: channels keep _ch{i} suffixes ---
    (d / 'multi' / 'in').mkdir(parents=True)
    ch0 = rng.integers(0, 256, (40, 40), np.uint8)
    ch1 = rng.integers(0, 256, (40, 40), np.uint8)
    cv2.imwrite(str(d / 'multi' / 'in' / 'b_ch0.png'), ch0)
    cv2.imwrite(str(d / 'multi' / 'in' / 'b_ch1.png'), ch1)
    cv2.imwrite(str(d / 'multi' / 'in' / 'b_mask.png'), mask)
    meta2 = extract_patches_from_sample('b', [d / 'multi' / 'in' / 'b_ch0.png', d / 'multi' / 'in' / 'b_ch1.png'], d / 'multi' / 'in' / 'b_mask.png', d / 'multi_out', cfg)
    assert len(meta2) >= 1
    assert len(list((d / 'multi_out' / 'images').glob('*_ch0.png'))) == len(meta2)
    assert len(list((d / 'multi_out' / 'images').glob('*_ch1.png'))) == len(meta2)
    assert len(list((d / 'multi_out' / 'masks').glob('*.png'))) == len(meta2)

In [ ]:
# test: multilabel patch extraction -- 'stacked' and 'per_class_files', overlap preserved through cropping
with tempfile.TemporaryDirectory() as d:
    d = Path(d)
    (d / 'src' / 'images').mkdir(parents=True)
    (d / 'src' / 'masks').mkdir(parents=True)
    img = rng.integers(0, 256, (40, 40), np.uint8)
    cv2.imwrite(str(d / 'src' / 'images' / 'a.png'), img)

    stacked = np.zeros((40, 40, 3), np.uint8)
    stacked[8:32, 8:32, 0] = 255   # chip
    stacked[16:24, 16:24, 1] = 255  # void, inside chip
    stacked[10:30, 10:30, 2] = 255  # lifted
    cv2.imwrite(str(d / 'src' / 'masks' / 'a.png'), stacked)
    cfg = PatchConfig(patch_size=20, stride=20, min_fg_pixels=1, max_empty_fraction=0.0)
    meta = extract_patches_from_sample(
        'a', [d / 'src' / 'images' / 'a.png'], str(d / 'src' / 'masks' / 'a.png'), d / 'stacked_out', cfg,
        mask_mode='stacked', mask_channels=3,
    )
    assert len(meta) >= 1
    pid = meta[0]['sample_id']
    saved = cv2.imread(str(d / 'stacked_out' / 'masks' / f'{pid}.png'), cv2.IMREAD_UNCHANGED)
    assert saved.shape == (20, 20, 3)
    assert set(meta[0]['class_pixels'].keys()) == {0, 1, 2}

    chip = np.zeros((40, 40), np.uint8); chip[8:32, 8:32] = 255
    void = np.zeros((40, 40), np.uint8); void[16:24, 16:24] = 255
    lifted = np.zeros((40, 40), np.uint8); lifted[10:30, 10:30] = 255
    for name, m in [('chip', chip), ('void', void), ('lifted', lifted)]:
        cv2.imwrite(str(d / 'src' / 'masks' / f'a_{name}.png'), m)
    meta2 = extract_patches_from_sample(
        'a', [d / 'src' / 'images' / 'a.png'], None, d / 'pcf_out', cfg,
        mask_mode='per_class_files',
        mask_paths=[str(d / 'src' / 'masks' / f'a_{n}.png') for n in ('chip', 'void', 'lifted')],
        class_names=['chip', 'void', 'lifted'],
    )
    assert len(meta2) >= 1
    pid2 = meta2[0]['sample_id']
    for name in ('chip', 'void', 'lifted'):
        assert (d / 'pcf_out' / 'masks' / f'{pid2}_{name}.png').exists()
    assert set(meta2[0]['class_pixels'].keys()) == {'chip', 'void', 'lifted'}
    # same source geometry -> same fg_pixels total (chip+void+lifted) across both mask_modes for the same patch
    assert meta[0]['fg_pixels'] == meta2[0]['fg_pixels']
print('extract_patches_from_sample ✓  multilabel: stacked + per_class_files, overlap preserved through cropping')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()